In [25]:
import os
import numpy as np
import librosa
import librosa.display
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import warnings
warnings.filterwarnings('ignore')


In [26]:
def generate_mfcc(label, num_samples=100):
    data = []
    for _ in range(num_samples):
        y = np.random.randn(22050) if label == "normal" else np.random.randn(22050) * 2
        mfcc = librosa.feature.mfcc(y=y, sr=22050, n_mfcc=40)
        mfcc = mfcc[:, :44]  # Pad/crop to fixed size
        data.append((mfcc, label))
    return data

normal_data = generate_mfcc("normal", 100)
anomaly_data = generate_mfcc("anomaly", 100)
dataset = normal_data + anomaly_data
np.random.shuffle(dataset)

print(f"Generated {len(dataset)} synthetic MFCC samples.")

Generated 200 synthetic MFCC samples.


In [27]:
X = []
y = []

for features, label in dataset:
    X.append(features)
    y.append(label)

X = np.array(X)
X = X[..., np.newaxis]  # Add channel dimension
y = LabelEncoder().fit_transform(y)
y = to_categorical(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X shape: {X.shape}, y shape: {y.shape}")


X shape: (200, 40, 44, 1), y shape: (200, 2)


In [28]:
model = Sequential([
    Conv2D(16, (3,3), activation='relu', input_shape=X.shape[1:]),
    MaxPooling2D(2,2),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(2, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                        ┃ Output Shape               ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)                   │ (None, 38, 42, 16)         │             160 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)      │ (None, 19, 21, 16)         │               0 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                   │ (None, 17, 19, 32)         │           4,640 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)      │ (None, 8, 9, 32)           │               0 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                 │ (None, 2304)               │               0 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                     │ (None, 64)                 │         147,520 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                 │ (None, 64)                 │               0 │
├─────────────────────────────────────┼────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                     │ (None, 2)                  │             130 │
└─────────────────────────────────────┴────────────────────────────┴─────────────────┘

 Total params: 152,450 (595.51 KB)

 Trainable params: 152,450 (595.51 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test), batch_size=16)


Epoch 1/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 90ms/step - accuracy: 0.6058 - loss: 1.3405 - val_accuracy: 0.8500 - val_loss: 0.4102
Epoch 2/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.7245 - loss: 0.5472 - val_accuracy: 0.8750 - val_loss: 0.3836
Epoch 3/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8694 - loss: 0.3605 - val_accuracy: 0.8500 - val_loss: 0.3184
Epoch 4/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8669 - loss: 0.2933 - val_accuracy: 0.7500 - val_loss: 0.4547
Epoch 5/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9131 - loss: 0.2527 - val_accuracy: 0.9750 - val_loss: 0.1937
Epoch 6/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9921 - loss: 0.1116 - val_accuracy: 0.9750 - val_loss: 0.1679
Epoch 7/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9946 - loss: 0.0686 - val_accuracy: 0.9750 - val_loss: 0.0919
Epoch 8/10
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9975 - loss: 0.0559 - val_accuracy: 0.9750 - v

In [30]:
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc*100:.2f}%")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 1.0000 - loss: 0.0561
Test Accuracy: 100.00%


In [31]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("audio_classifier.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved as 'audio_classifier.tflite'")


INFO:tensorflow:Assets written to: C:\Users\user\AppData\Local\Temp\tmp3ue0rgt9\assets


INFO:tensorflow:Assets written to: C:\Users\user\AppData\Local\Temp\tmp3ue0rgt9\assets


Saved artifact at 'C:\Users\user\AppData\Local\Temp\tmp3ue0rgt9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 44, 1), dtype=tf.float32, name='keras_tensor_18')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2163869955280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2163869943952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2163869947600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2163869941840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2163869956048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2163869957392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2163845867344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2163845868304: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model saved as 'audio_classifier.tflite'


In [34]:
file_path = 'sound.wav'
y, sr = librosa.load(file_path, sr=22050, duration=5.0)

# Preprocess
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
mfcc = mfcc[:, :44]
mfcc = mfcc[np.newaxis, ..., np.newaxis]  # reshape for model

# Predict
pred = model.predict(mfcc)
print("Prediction:", "Anomaly" if pred[0][1] > 0.5 else "Normal")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 842ms/step
Prediction: Anomaly
